In [1]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("./milvus_rag.db")

In [2]:
from pymilvus import CollectionSchema, DataType, FieldSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(
    name="id", dtype=DataType.INT64, is_primary=True, description="Primary id"
)
text = FieldSchema(
    name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text"
)
embedding_text = FieldSchema(
    "embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=VECTOR_LENGTH,
    description="Embedded text",
)

fields = [id_field, text, embedding_text]

schema = CollectionSchema(
    fields=fields,
    auto_id=True,
    enable_dynamic_field=True,
    description="RAG Texts collection",
)

In [3]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64},  # lower values for speed
)

milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 0, 'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True}


In [4]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [5]:
# download data
import os
import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)


download_pdf_data(pdf_url, file_name)

In [6]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)

In [7]:
# vectorize data

import torch
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append(
            {"page_num": page_num, "embedding": embeddings[page_num].tolist()}
        )

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)


model_name = "ipipan/silver-retriever-base-v1.1"
device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

I0000 00:00:1793556851.750270 5281106 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [8]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with (
        open(os.path.join(data_dir, file_json), "r") as t_f,
        open(os.path.join(data_dir, embeddings_json), "r") as e_f,
    ):
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data = [d["text"] for d in text_data]
        embedding_data = [d["embedding"] for d in embedding_data]

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text": text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [9]:
# search


def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"],
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [10]:
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_KEY is not None, "GEMINI_API_KEY environment variable is not set."
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"


def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [11]:
def build_prompt(context: str, query: str, style: str = "") -> str:
    prompt = f"""
Use the following context to answer the question. If the answer is not contained
within the context, admit that and don't try to make up an answer.
{style}
Context: {context}
Question: {query}"""
    return prompt


def rag(model, query: str, style: str = "") -> str:
    # having all prepared functions, you can combine them together and try to build your own RAG!
    context = search(model, query=query)
    context_str = " ".join([doc["text"] for doc in context[0]])
    prompt = build_prompt(context=context_str, query=query, style=style)
    response = generate_response(prompt)
    assert response is not None, "Failed to generate response from the model."
    return response

In [12]:
rag(model, query="Podaj przykład generatywnego modelu dźwięku")

'Przykładem generatywnego modelu dźwięku jest WaveGAN.'

In [13]:
print(rag(model, query="Co to jest maszyna reaktywna?"))

Maszyna reaktywna to SI, którą stworzono do rozwiązania określonego problemu. Nie posiada pamięci, a w związku z tym nie jest w stanie uczyć się. Przykładem jest Deep Blue, program szachowy IBM, który pokonał Garry’ego Kasparova w latach 90.


In [14]:
rag(
    model,
    query="Co to jest maszyna reaktywna?",
    style="Lista 3 bullet pointów kazdy z nich powinien zaczynać się od emoji",
)

'Maszyna reaktywna to SI, którą stworzono do rozwiązania określonego problemu. Nie posiada pamięci, a w związku z tym nie jest w stanie się uczyć.'

In [15]:
rag(
    model,
    query="Co to jest maszyna reaktywna?",
    style="Jednym zdaniem",
)

'Maszyna reaktywna to SI, którą stworzono do rozwiązania określonego problemu.'

In [16]:
rag(
    model,
    query="Co to jest maszyna reaktywna?",
    style="Answer like an gen alpha, brain rotted teenager",
)

'OMG, maszyna reaktywna? To taka SI, co jest literally stworzona do ogarnięcia tylko jednego problemu, wiesz? Jakby, ma jedną robotę i tyle. Totalnie nie ma pamięci, więc nie ogarnia nauki, cringe. Jak Deep Blue, ten komp od szachów, co pokonał Kasparova w latach 90. To taki boomer AI, no.'

In [17]:
rag(
    model,
    query="Co to jest maszyna reaktywna?",
    style="Answer in pirate speak",
)

"Arrr, me hearty! A reactive machine, ye ask? 'Tis a type o' AI, built solely to solve a particular problem, see? It's got no memory in its locker, no sir, and so it can't learn a single thing. A fine example be that scallywag Deep Blue, the chess program from IBM that outsmarted Garry Kasparov back in the nineties! Arrr!"

Notes:
- I passed the style only to the prompt, so search still gets the plain question. All four tweaks found the right page and the answers stuck to it.
- One sentence, gen-alpha and pirate all worked. The emoji bullets didn't, the model just wrote plain text.
- Before that I had the style inside the query. The emoji and gen-alpha runs then pulled the wrong pages and the model said it couldn't find the answer.